# RQ3 disability strand + age strand -- supplementary mixed-effects model (partial pooling by borough)


## 1. Setup and data (disability strand)


In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

DISABILITY_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed")
AGE_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\q3")

dis_overall = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_overall_all_years.csv')
dis_overall['LA_2023'] = dis_overall['LA_2023'].astype('Int64').astype(str)
dis_overall = dis_overall.sort_values(['LA_2023', 'disability_group', 'year']).reset_index(drop=True)


## 2. Feature construction (disability strand)



In [2]:
target_cols = ['inactive_rate', 'fairly_active_rate', 'active_rate']

grouped = dis_overall.groupby(['LA_2023', 'disability_group'], sort=False)
for col in target_cols:
    dis_overall[f'{col}_lag1'] = grouped[col].shift(1)

dis_overall['time_trend'] = dis_overall['year'] / dis_overall['year'].max()
dis_overall['is_covid_year'] = dis_overall['year'].isin([5, 6]).astype(int)


## 3. Train / test split (disability strand)


In [7]:
required_cols = target_cols + ['disability_group', 'time_trend', 'is_covid_year'] + [f'{c}_lag1' for c in target_cols]
trainval = dis_overall[dis_overall['year'] < 8].dropna(subset=required_cols).reset_index(drop=True)
test = dis_overall[dis_overall['year'] == 8].dropna(subset=required_cols).reset_index(drop=True)

## 4. Fit random-intercept models per target (disability strand)



In [8]:
mixedlm_results = {}
mixedlm_models = {}

for col in target_cols:
    formula = f'{col} ~ disability_group + time_trend + is_covid_year + {col}_lag1'
    model = smf.mixedlm(formula, data=trainval, groups=trainval['LA_2023'].to_numpy())
    fitted = model.fit()
    mixedlm_models[col] = fitted

    pred = fitted.predict(test)
    mae = (pred - test[col]).abs().mean()
    rmse = ((pred - test[col]) ** 2).mean() ** 0.5
    mixedlm_results[col] = {'mae': round(mae, 4), 'rmse': round(rmse, 4)}
    print(col, mixedlm_results[col])
    print(fitted.summary())
    print()


d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


inactive_rate {'mae': np.float64(0.1508), 'rmse': np.float64(0.198)}
                         Mixed Linear Model Regression Results
Model:                      MixedLM          Dependent Variable:          inactive_rate
No. Observations:           2990             Method:                      REML         
No. Groups:                 32               Scale:                       0.0393       
Min. group size:            87               Log-Likelihood:              515.5641     
Max. group size:            96               Converged:                   Yes          
Mean group size:            93.4                                                       
---------------------------------------------------------------------------------------
                                            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                    0.468    0.020  22.960 0.000  0

d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 5. Borough random intercepts -- partial pooling check (disability strand)



In [9]:
random_intercepts = {}
for col in target_cols:
    re = mixedlm_models[col].random_effects
    random_intercepts[col] = pd.Series({k: v.iloc[0] for k, v in re.items()}, name=f'{col}_intercept')

borough_intercepts = pd.concat(random_intercepts.values(), axis=1).reset_index()
borough_intercepts = borough_intercepts.rename(columns={'index': 'LA_2023'})
borough_intercepts.sort_values('active_rate_intercept')


,LA_2023,inactive_rate_intercept,fairly_active_rate_intercept,active_rate_intercept
14,17,0.000046,0.059067,-0.067106
5,122,0.070823,-0.003576,-0.065333
29,8,0.057261,0.005145,-0.062649
16,196,0.048602,0.006748,-0.054308
15,171,0.049208,0.004381,-0.053552
1,109,0.055225,-0.002523,-0.052559
24,30,0.053506,-0.002131,-0.050393
0,107,0.006799,0.037158,-0.048571
19,255,0.028877,0.016979,-0.047567
27,68,0.039138,0.002408,-0.041100


## 6. Age strand -- random-intercept model (parallel to disability strand above)



In [10]:
age_overall = pd.read_csv(AGE_DATA_DIR / 'q3_age_overall_activity_level_panel.csv')
age_overall['LA_2023'] = age_overall['LA_2023'].astype('Int64').astype(str)
age_overall = age_overall.sort_values(['LA_2023', 'age_group', 'year']).reset_index(drop=True)


## 7. Feature construction (age strand)


In [11]:
age_target_cols = ['overall_inactive_rate', 'overall_fairly_active_rate', 'overall_active_rate']

age_grouped = age_overall.groupby(['LA_2023', 'age_group'], sort=False)
for col in age_target_cols:
    age_overall[f'{col}_lag1'] = age_grouped[col].shift(1)

age_overall['time_trend'] = age_overall['year'] / age_overall['year'].max()
age_overall['is_covid_year'] = age_overall['year'].isin([5, 6]).astype(int)


## 8. Train / test split (age strand)


In [12]:
age_required_cols = age_target_cols + ['age_group', 'time_trend', 'is_covid_year'] + [f'{c}_lag1' for c in age_target_cols]
age_trainval = age_overall[age_overall['year'] < 8].dropna(subset=age_required_cols).reset_index(drop=True)
age_test = age_overall[age_overall['year'] == 8].dropna(subset=age_required_cols).reset_index(drop=True)

## 9. Fit random-intercept models per target (age strand)



In [17]:
age_mixedlm_results = {}
age_mixedlm_models = {}

for col in age_target_cols:
    formula = f'{col} ~ age_group + time_trend + is_covid_year + {col}_lag1'
    model = smf.mixedlm(formula, data=age_trainval, groups=age_trainval['LA_2023'].to_numpy())
    fitted = model.fit()
    age_mixedlm_models[col] = fitted

    pred = fitted.predict(age_test)
    mae = (pred - age_test[col]).abs().mean()
    rmse = ((pred - age_test[col]) ** 2).mean() ** 0.5
    age_mixedlm_results[col] = {'mae': round(mae, 4), 'rmse': round(rmse, 4)}
    print(col, age_mixedlm_results[col])
    print(fitted.summary())
    print()

d:\Anaconda\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
d:\Anaconda\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(
d:\Anaconda\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Anaconda\Lib\site-packages\statsmodels\regress

overall_inactive_rate {'mae': np.float64(0.0921), 'rmse': np.float64(0.131)}
                Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: overall_inactive_rate
No. Observations:   1536    Method:             REML                 
No. Groups:         32      Scale:              0.0132               
Min. group size:    48      Log-Likelihood:     1073.5463            
Max. group size:    48      Converged:          No                   
Mean group size:    48.0                                             
---------------------------------------------------------------------
                           Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------
Intercept                   0.191    0.015 12.819 0.000  0.162  0.220
age_group[T.25-34]         -0.002    0.012 -0.140 0.888 -0.025  0.021
age_group[T.35-44]          0.024    0.012  2.005 0.045  0.001  0.047
age_group[T.45-54]          0

d:\Anaconda\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
d:\Anaconda\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(


overall_active_rate {'mae': np.float64(0.0962), 'rmse': np.float64(0.1354)}
               Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  overall_active_rate
No. Observations:  1536     Method:              REML               
No. Groups:        32       Scale:               0.0127             
Min. group size:   48       Log-Likelihood:      1109.7874          
Max. group size:   48       Converged:           No                 
Mean group size:   48.0                                             
--------------------------------------------------------------------
                         Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------
Intercept                 0.638    0.022  28.472 0.000  0.594  0.682
age_group[T.25-34]       -0.005    0.011  -0.430 0.667 -0.027  0.018
age_group[T.35-44]       -0.034    0.012  -2.978 0.003 -0.057 -0.012
age_group[T.45-54]       -0.033    0.012  -

d:\Anaconda\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 14.530833
  warnings.warn(msg, ConvergenceWarning)
d:\Anaconda\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Note: overall_fairly_active_rate converged normally. The other two targets, overall_inactive_rate and overall_active_rate, failed to converge. Numerical-scale issues were ruled out (standardising the lag1 predictor gave no improvement). The more likely explanation is that the age_group fixed effect has very strong explanatory power, which compresses the estimable space left for borough-level random effects. The coefficients and standard errors from these two models are reported for reference only and are not used as formal conclusions.

## 10. Borough random intercepts -- partial pooling check (age strand)



In [14]:
age_random_intercepts = {}
for col in age_target_cols:
    re = age_mixedlm_models[col].random_effects
    age_random_intercepts[col] = pd.Series({k: v.iloc[0] for k, v in re.items()}, name=f'{col}_intercept')

age_borough_intercepts = pd.concat(age_random_intercepts.values(), axis=1).reset_index()
age_borough_intercepts = age_borough_intercepts.rename(columns={'index': 'LA_2023'})
age_borough_intercepts.sort_values('overall_active_rate_intercept')


,LA_2023,overall_inactive_rate_intercept,overall_fairly_active_rate_intercept,overall_active_rate_intercept
15,171,0.089363,0.005173,-0.089998
29,8,0.097757,-0.002015,-0.083691
24,30,0.056825,0.002874,-0.056706
16,196,0.045439,0.007381,-0.055119
7,129,0.041611,0.000097,-0.037421
4,117,0.026730,0.006636,-0.036696
20,272,0.042759,-0.000782,-0.036307
21,279,0.008326,0.010899,-0.028088
6,126,0.028321,-0.000834,-0.024433
5,122,0.023433,0.000112,-0.020961
